# 암종 분류 v6 (v3 + 쌍둥이 규칙 + 표기 통일): 이진 행렬 + 파생변수 + 인프레임 핫스팟 + 동반변이 콤보 + XGB/LR 앙상블 (5-fold CV)

**v2 대비 변경점**
- 인프레임 변이(`EGFR E746_A750del` 등)를 핫스팟 토큰으로 추가 (`gene|inframe`, `gene|위치_inframe`)
- 가이드 기반 동반변이 콤보 피처 (`cb_*`), TMB 기준 5 / 30
- **EM 사전확률 보정 제거** (macroF1을 0.46 -> 0.21로 망가뜨림). COAD 확률 보정은 검토 후 이득이 기준 미만(미적용)이고 test 분포 정보가 필요해서 **삭제**
- 실행 순서 버그 수정: 예전 prior 셀이 CV보다 앞에 있어 `oof_ens`가 정의되기 전에 쓰였음 -> 앙상블 이후로 이동
- CV를 함수(`run_cv`)로 묶고, 콤보 ON/OFF 비교(ablation) 셀 추가

**패키지**: `pip install pandas numpy scikit-learn xgboost`


**v5 변경점**: 쌍둥이(형제 라벨) 규칙 추가 (5-2절). KIRC↔KIPAN, LGG↔GBMLGG 는 같은 환자가 두 코호트에 중복 등록된 경우가 있어, train에 짝 라벨이 있는 프로필이 test에 나오면 짝 암종으로 교정한다. (최소빈도는 v3 그대로 5)

**v6 주요 변경점**: 변이 표기 정규화 (`NORMALIZE_TOKENS`, `canon`)
- HGVS 규약상 같은 변이가 `R1450X` / `R1450*` / `p.Arg1450Ter` 등으로 달리 적힐 수 있어, 표기와 무관하게 같은 변이가 같은 피처가 되도록 정규화하고 한 칸 안의 중복 표기를 제거한다.
- 모든 연산은 토큰/행 하나만 보는 독립 연산이며 train/test에 동일하게 적용된다. test 통계는 사용하지 않는다.
- 단, train에는 구식 표기가 0건이라 train만으로 효과를 검증할 수 없다. 운영진 확인 전까지는 `NORMALIZE_TOKENS=False` 제출본(`nonorm`)을 함께 준비한다.
- **v7 보강(canon)**: 괄호 `p.(Arg1450Ter)`, 대문자 `P.`, 소문자 `x`/`ter`, 대소문자 섞인 3글자 표기도 같은 변이로 통일. train에는 해당 표기가 0건이라 기존 결과는 변하지 않는다.

**v8 변경점 (TwinRuleRefine.md 반영)**: 후처리 규칙으로 점수를 짜내는 대신 데이터 정의와 분할 단계에서 푼다.
- **그룹 분할 CV** (`GROUP_CV=True`): 변이 프로필이 같은 행(쌍둥이·중복)은 같은 fold에 묶는다. 같은 프로필이 학습/검증에 갈라져 OOF가 부풀려지는 것을 막는다. (`StratifiedGroupKFold`, 그룹 = 프로필 해시)
- **프로필 중복·라벨 충돌 진단 셀** 추가 (4-0절): 데이터 큐레이션의 근거.
- **쌍둥이 규칙 기본 OFF** (`USE_TWIN=False`): 분석가가 라벨 구조를 이용해 예측을 덮어쓰는 후처리라 확장성·규정 리스크가 크다. 진단용으로만 남겼고, 켜려면 플래그와 시뮬 이득 조건을 모두 만족해야 한다.
- (선택) `PROFILE_WEIGHT`: 같은 프로필 행이 학습을 지배하지 않도록 1/그룹크기 가중치. 기본 OFF, 효과는 CV로 확인 필요.
- 주의: 그룹 분할로 바뀌면 OOF 점수는 이전(무작위 분할)보다 낮아질 수 있다. 이전 점수가 중복 덕분에 부풀려져 있었다는 뜻이지 모델이 나빠진 것이 아니다.


In [1]:
import os, re, warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
warnings.filterwarnings("ignore")

CFG = dict(
    DATA_DIR=os.environ.get("DATA_DIR", "../../data"),
    SEED=42,
    N_FOLDS=int(os.environ.get("N_FOLDS", 5)),
    MIN_GENE_FREQ=5,        # train에서 변이 샘플이 이 값 미만인 유전자는 제외
    MIN_HOTSPOT_FREQ=5,     # gene|position 토큰 최소 빈도
    MIN_AA_FREQ=20,         # 아미노산 치환쌍 최소 빈도
    USE_AA_SUBS=True,       # 아미노산 치환 패턴 피처 on/off (CV로 효과 확인)
    USE_PATHWAYS=True,      # 경로 집계 피처 on/off
    USE_HOTSPOTS=True,      # gene|position, gene|trunc 토큰 on/off
    BALANCE=True,           # 클래스 불균형 보정 (sample weight / class_weight)
    XGB_N_EST=int(os.environ.get("XGB_N_EST", 2000)),
    LR_C=0.1,
    USE_COMBOS=True,        # 동반 변이 콤보 피처 on/off
    TMB_LOW=5,              # 극저변이 기준 (THCA/LAML 등)
    TMB_HIGH=30,            # 고변이 기준 (SKCM/UCEC/STES 등)
    BOOST_MIN_GAIN=0.003,   # 쌍둥이 규칙 등 후처리가 OOF macroF1을 이만큼 못 올리면 적용 안 함
    NORMALIZE_TOKENS=True,  # 변이 표기 통일(HGVS 구/신 표기) + 한 칸 안 중복 제거. 운영진 확인 전에는 False 제출본도 준비
    USE_TWIN=False,         # 쌍둥이(형제 라벨) 후처리 규칙. 기본 OFF: 라벨 구조를 이용한 덮어쓰기라 확장성/규정 리스크가 큼 (진단용으로만 사용)
    GROUP_CV=True,          # 같은 변이 프로필(쌍둥이·중복)을 같은 fold에 묶어 분할 -> OOF 누수 방지
    PROFILE_WEIGHT=False,   # 같은 프로필 행에 1/그룹크기 가중치 (중복 행이 학습을 지배하지 않게). CV로 확인 후 사용
    RUN_ABLATION=False,     # True면 콤보 ON/OFF CV 비교 (시간 2배)
)


## 1. 데이터 로드

In [2]:
D = CFG["DATA_DIR"]
train = pd.read_csv(f"{D}/train.csv").fillna("WT")
test = pd.read_csv(f"{D}/test.csv").fillna("WT")
sub = pd.read_csv(f"{D}/sample_submission.csv")

assert (test["ID"].values == sub["ID"].values).all(), "test와 sample_submission의 ID 순서가 다릅니다"
gene_cols = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
assert gene_cols == [c for c in test.columns if c != "ID"], "train/test 유전자 컬럼이 다릅니다"

le = LabelEncoder()
y = le.fit_transform(train["SUBCLASS"])
n_class = len(le.classes_)
print("train", train.shape, "classes", n_class)
print(train["SUBCLASS"].value_counts())

train (6201, 4386) classes 26
SUBCLASS
BRCA      786
KIPAN     515
GBMLGG    461
STES      379
KIRC      334
THCA      324
SKCM      276
PRAD      266
OV        253
LGG       229
HNSC      223
COAD      223
SARC      198
UCEC      198
LUAD      184
LUSC      178
LIHC      158
LAML      158
CESC      155
PCPG      147
TGCT      124
PAAD      120
BLCA      104
THYM       98
ACC        72
DLBC       38
Name: count, dtype: int64


In [3]:
train.head(5)

,ID,SUBCLASS,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
0,TRAIN_0000,KIPAN,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
1,TRAIN_0001,SARC,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
2,TRAIN_0002,SKCM,R895R,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
3,TRAIN_0003,KIRC,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
4,TRAIN_0004,GBMLGG,WT,WT,WT,WT,WT,WT,WT,WT,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT


## EDA

In [4]:
import pandas as pd
import numpy as np
import re

# 1. 데이터 로드
train = pd.read_csv(f"{D}/train.csv").fillna("WT")

print("="*50)
print("1. 기본 데이터 크기 및 암종(SUBCLASS) 개수 확인")
print("="*50)
print(f"Train 크기: {train.shape}")

# 암종(SUBCLASS) 개수 및 분포 확인
subclass_counts = train["SUBCLASS"].value_counts()
print(f"총 암종(Subclass) 개수: {len(subclass_counts)}개")
print("\n[상위 5개 암종 분포]")
print(subclass_counts.head())

# 유전자 컬럼 분리
gene_cols = [c for c in train.columns if c not in ("ID", "SUBCLASS")]
print(f"\n유전자(Feature) 컬럼 총 개수: {len(gene_cols)}개")


print("\n" + "="*50)
print("2. 샘플당 변이 유전자 수 분포 (TMB)")
print("="*50)
# WT가 아닌 유전자의 개수를 행별로 카운트
mut_matrix = (train[gene_cols] != "WT")
mut_counts_per_sample = mut_matrix.sum(axis=1)
print(mut_counts_per_sample.describe().round(2))


print("\n" + "="*50)
print("3. 한 셀에 변이가 여러 개 들어있는지 확인 (구분자 체크)")
print("="*50)
# WT가 아닌 셀들만 추출
all_vals = train[gene_cols].values.ravel()
non_wt_vals = all_vals[all_vals != "WT"]

non_wt_series = pd.Series(non_wt_vals)
# 쉼표(,), 콜론(:), 세미콜론(;), 파이프(|), 공백 등이 포함된 비율 확인
sep_pattern = r"[,:;|\s]"
has_sep = non_wt_series.str.contains(sep_pattern, regex=True).mean()
print(f"Non-WT 값 중 구분자(, : ; | 공백)가 포함된 비율: {has_sep:.4%}")


print("\n" + "="*50)
print("4. 변이 표기 패턴 종류 확인 (정규식 기반 상위 패턴)")
print("="*50)
# 숫자를 'N'으로 치환하여 대표적인 변이 표기 패턴 분석
pattern_samples = non_wt_series.str.replace(r"\d+", "N", regex=True)
print(pattern_samples.value_counts().head(15))


print("\n" + "="*50)
print("5. 유전자별 변이 빈도 (희소성 확인)")
print("="*50)
gene_mut_freq = mut_matrix.sum(axis=0)
print(f"변이가 단 1개라도 발생한 유전자 수: {(gene_mut_freq > 0).sum()} / {len(gene_cols)}")
print(f"대다수 유전자의 변이 빈도 요약 (상위/하위):")
print(gene_mut_freq.describe().round(2))

1. 기본 데이터 크기 및 암종(SUBCLASS) 개수 확인
Train 크기: (6201, 4386)
총 암종(Subclass) 개수: 26개

[상위 5개 암종 분포]
SUBCLASS
BRCA      786
KIPAN     515
GBMLGG    461
STES      379
KIRC      334
Name: count, dtype: int64

유전자(Feature) 컬럼 총 개수: 4384개

2. 샘플당 변이 유전자 수 분포 (TMB)
count    6201.00
mean       35.30
std       100.78
min         0.00
25%         7.00
50%        14.00
75%        27.00
max      2393.00
dtype: float64

3. 한 셀에 변이가 여러 개 들어있는지 확인 (구분자 체크)
Non-WT 값 중 구분자(, : ; | 공백)가 포함된 비율: 10.0625%

4. 변이 표기 패턴 종류 확인 (정규식 기반 상위 패턴)
LNL    7409
ENK    6790
SNS    4517
RNH    4441
RNC    4141
RNQ    4127
PNP    4037
ANA    3935
TNT    3841
ANT    3712
DNN    3685
INI    3680
ANV    3652
GNG    3581
PNS    3536
Name: count, dtype: int64

5. 유전자별 변이 빈도 (희소성 확인)
변이가 단 1개라도 발생한 유전자 수: 4230 / 4384
대다수 유전자의 변이 빈도 요약 (상위/하위):
count    4384.00
mean       49.93
std        57.64
min         0.00
25%        20.00
50%        36.00
75%        61.00
max      1770.00
dtype: float64


## 2. 변이 표기 형식 확인 (파싱 규칙이 맞는지 눈으로 검증)

In [5]:
sample = pd.Series(train[gene_cols].iloc[:1000].to_numpy(dtype=object).ravel())
nonwt = sample[sample != "WT"]
print(f"non-WT 비율: {len(nonwt)/len(sample):.4%}")
print("한 셀에 구분자(, : ; | 공백)가 있는 비율:", nonwt.str.contains(r"[,:;|\s]").mean().round(4))
print(nonwt.str.replace(r"\d+", "N", regex=True).value_counts().head(25))

non-WT 비율: 0.8337%
한 셀에 구분자(, : ; | 공백)가 있는 비율: 0.0873
LNL    1236
ENK    1125
SNS     777
RNH     752
ANA     738
RNC     724
PNP     711
RNQ     696
TNT     665
ANT     642
GNG     636
ANV     629
FNF     614
PNL     610
DNN     609
PNS     593
INI     561
VNV     555
RNR     492
RNW     483
SNF     437
RN*     410
EN*     388
GNR     341
KNN     328
Name: count, dtype: int64


## 3. 피처 생성

- `g_*` : 유전자별 변이 여부 (이진)
- TMB 계열 : 변이 유전자 수, log, 이벤트 수
- 변이 유형 : missense / silent / nonsense / frameshift / splice / inframe 개수와 비율
- `hs_*` : 핫스팟(`유전자|원래아미노산+위치`) 및 truncating(`유전자|trunc`) 이진
- `aa_*` : 아미노산 치환쌍 비율
- `pw_*` : 경로별 변이 유전자 수 (아래 `PATHWAYS` 사전은 초안이니 자유롭게 수정)

In [6]:
def combo_features(ex):
    mut = ex["mut"]; n = mut.shape[0]
    n_genes = mut.sum(1)
    low, high = n_genes <= CFG["TMB_LOW"], n_genes >= CFG["TMB_HIGH"]

    def G(name):   # 유전자 변이 여부 (없는 유전자는 0 벡터 — 엉뚱한 컬럼 안 가져옴)
        j = gene_pos.get(name)
        return mut[:, j].copy() if j is not None else np.zeros(n, bool)

    tok_rows = {}
    for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
        tok_rows.setdefault(nm, []).append(r)
    def T(name):   # 핫스팟/절단형 토큰 보유 여부 (MIN_HOTSPOT_FREQ와 무관하게 직접 계산)
        v = np.zeros(n, bool)
        if name in tok_rows: v[tok_rows[name]] = True
        return v

    f = {}
    # COAD
    f["COAD_APCtr_TP53"]  = T("APC|trunc") & G("TP53")
    f["COAD_APCtr_KRAS"]  = T("APC|trunc") & (G("KRAS") | G("FBXW7"))
    f["COAD_score"]       = T("APC|trunc").astype(int) + G("TP53") + G("KRAS") + G("FBXW7") + G("PIK3CA")
    # KIRC / KIPAN
    f["KIRC_VHL_PBRM1"]   = G("VHL") & G("PBRM1")
    f["KIRC_3p_score"]    = G("VHL").astype(int) + G("PBRM1") + G("SETD2") + G("BAP1")
    # UCEC
    f["UCEC_PTEN_CTNNB1"] = G("PTEN") & G("CTNNB1")
    f["UCEC_PTEN_PIK3CA"] = G("PTEN") & G("PIK3CA")
    f["UCEC_highTMB_PTEN"]= high & G("PTEN")
    # PAAD
    f["PAAD_score"]       = T("KRAS|G12").astype(int) + G("TP53") + T("CDKN2A|trunc") + G("SMAD4")
    # LGG vs GBMLGG
    f["LGG_IDH1_TP53_ATRX"] = T("IDH1|R132") & G("TP53") & G("ATRX")
    f["LGG_IDH1_clean"]   = T("IDH1|R132") & ~(G("PTEN") | G("EGFR"))
    f["GBM_PTEN_EGFR"]    = G("PTEN") & G("EGFR")
    f["GBM_noIDH1_signal"]= (G("PTEN") | G("EGFR")) & ~G("IDH1")
    # BRAF V600 + TMB (THCA vs SKCM vs COAD)
    f["THCA_BRAF_lowTMB"] = T("BRAF|V600") & low
    f["SKCM_BRAF_highTMB"]= T("BRAF|V600") & high
    f["SKCM_NRAS_highTMB"]= T("NRAS|Q61") & high
    f["COAD_BRAF_APC"]    = T("BRAF|V600") & T("APC|trunc")
    # 폐암 / 위식도 / 간 / 난소 / 백혈병
    f["LUAD_EGFR_noTP53"] = G("EGFR") & ~G("TP53")
    f["LUSC_TP53_noEGFR_highTMB"] = G("TP53") & ~G("EGFR") & high
    f["STES_highTMB_TP53_noAPC"]  = high & G("TP53") & ~T("APC|trunc")
    f["LIHC_CTNNB1_noAPC"]= G("CTNNB1") & ~G("APC")
    f["OV_TP53_lowTMB"]   = G("TP53") & (n_genes <= 20)
    f["LAML_NPM1_lowTMB"] = G("NPM1") & low

    names = list(f.keys())
    return np.c_[[np.asarray(f[k], np.float32) for k in names]].T, names

In [7]:
AA = "ACDEFGHIKLMNPQRSTVWY"
AA_IDX = {a: i for i, a in enumerate(AA)}
TYPES = ["missense", "silent", "nonsense", "frameshift", "splice", "inframe", "other"]
TYPE_IDX = {t: i for i, t in enumerate(TYPES)}
SEP = re.compile(r"[,:;|\s]+")
AA3 = {"Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E","Gly":"G","His":"H","Ile":"I",
       "Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V","Ter":"*"}

_A3 = "|".join(AA3)   # Ala|Arg|...|Ter
# 단순 치환형(예: Arg1450Ter, r1450x, ARG1450TER, p.(Arg1450Ter))은 통째로 매칭해서 대소문자까지 한 번에 통일
_SIMPLE = re.compile(rf"^({_A3}|[A-Z])(\d+)({_A3}|[A-Z]|\*)$", re.I)

def _aa1(x, last=False):
    """아미노산 표기 하나 -> 1글자. 마지막 자리의 X/x/Ter는 종결 코돈(*)으로 읽는다."""
    if x == "*":
        return x
    if len(x) == 3:
        return AA3.get(x.capitalize(), x)
    x = x.upper()
    return "*" if (last and x == "X") else x

def canon(tok):
    """변이 표기 1개를 한 가지 모양으로 통일한다 (그 토큰만 보고 바꿈 = 행 독립, train/test 동일 적용).
    p.Arg1450Ter / p.(Arg1450Ter) / R1450X / r1450x / R1450* -> R1450*   (옛 HGVS의 X도 종결 코돈으로 읽는 기존 동작 유지)
    v7 보강: 괄호(HGVS 예측 표기), 대문자 P., 소문자 x/ter, 대소문자 섞인 3글자 표기도 같은 변이로 통일."""
    t = tok.strip()
    t = re.sub(r"^\(?[pP]\.", "", t)      # p. / P. / (p. 접두 제거
    t = t.strip("()")                       # p.(Arg1450Ter) 같은 괄호 제거
    m = _SIMPLE.match(t)
    if m:
        a, pos, b = m.groups()
        a1, b1 = _aa1(a), _aa1(b, last=True)
        if a1 != "*":                       # 종결 코돈이 시작 자리에 오는 표기는 건드리지 않는다
            return f"{a1}{pos}{b1}"
    t = re.sub(r"[A-Z][a-z]{2}", lambda m: AA3.get(m.group(), m.group()), t)   # 3글자 아미노산 -> 1글자 (fs/del 등 복합 표기용)
    t = re.sub(r"^([A-Z])(\d+)[Xx]$", r"\1\2*", t)                             # R1450X -> R1450*
    return t

def cell_tokens(cell):
    """셀 하나 -> 변이 토큰 목록. NORMALIZE_TOKENS면 표기를 통일하고 같은 변이의 중복 표기를 하나로 합친다."""
    toks = [t for t in SEP.split(str(cell)) if t and t != "WT"]
    if CFG["NORMALIZE_TOKENS"]:
        toks = list(dict.fromkeys(canon(t) for t in toks))
    return toks
RE_MIS = re.compile(r"^(?:p\.)?([A-Z])(\d+)([A-Z])$")
RE_NON = re.compile(r"^(?:p\.)?[A-Z]\d+(\*|X|Ter)$")

def parse_token(tok):
    """변이 표기 1개 -> (유형, 원래aa, 위치, 바뀐aa). 표기가 다르면 여기 규칙을 수정하세요."""
    low = tok.lower()
    m = re.search(r"\d+", tok)
    pos = m.group() if m else ""
    if "fs" in low or "frameshift" in low:
        return "frameshift", "", pos, ""
    if "splice" in low:
        return "splice", "", pos, ""
    if "del" in low or "ins" in low or "dup" in low:
        return "inframe", "", pos, ""
    if RE_NON.match(tok):
        return "nonsense", "", pos, ""
    m = RE_MIS.match(tok)
    if m:
        fa, p, ta = m.groups()
        return ("silent" if fa == ta else "missense"), fa, p, ta
    return "other", "", pos, ""

def extract(df):
    arr = df[gene_cols].to_numpy(dtype=object)
    mut = arr != "WT"
    n = len(df)
    type_cnt = np.zeros((n, len(TYPES)), np.float32)
    aa_cnt = np.zeros((n, 400), np.float32)
    tok_rows, tok_names = [], []
    rr, cc = np.nonzero(mut)
    for r, c in zip(rr, cc):
        gene = gene_cols[c]
        for tok in cell_tokens(arr[r, c]):
            typ, fa, pos, ta = parse_token(tok)
            type_cnt[r, TYPE_IDX[typ]] += 1
            if typ == "missense":
                tok_rows.append(r); tok_names.append(f"{gene}|{fa}{pos}")
                if fa in AA_IDX and ta in AA_IDX:
                    aa_cnt[r, AA_IDX[fa] * 20 + AA_IDX[ta]] += 1
            elif typ in ("nonsense", "frameshift", "splice"):
                tok_rows.append(r); tok_names.append(f"{gene}|trunc")
            elif typ == "inframe":                       # ← 추가
                tok_rows.append(r); tok_names.append(f"{gene}|inframe")        # 유전자 단위
                tok_rows.append(r); tok_names.append(f"{gene}|{pos}_inframe")  # 위치 단위 (EGFR 746_inframe 등)
    return dict(mut=mut, type_cnt=type_cnt, aa_cnt=aa_cnt,
                tok_rows=np.array(tok_rows, dtype=np.int64), tok_names=tok_names)

ex_tr, ex_te = extract(train), extract(test)
print("파싱 완료. 유형별 총 개수(train):", dict(zip(TYPES, ex_tr["type_cnt"].sum(0).astype(int))))
# ---- 표기 통일 동작 확인 ----
assert canon("R1450X") == "R1450*" == canon("p.Arg1450Ter") == canon("p.R1450*")
assert canon("E746_A750del") == "E746_A750del" and canon("S221Kfs") == "S221Kfs"
assert parse_token(canon("R1450X"))[0] == "nonsense"
# v7 보강 케이스
assert canon("p.(Arg1450Ter)") == canon("R1450x") == canon("ARG1450TER") == canon("P.Arg1450X") == "R1450*"
assert canon("p.Val600Glu") == canon("(V600E)") == canon("v600e") == "V600E"
assert canon("p.Arg97ProfsTer23") == "R97Pfs*23"
assert canon("X123R") == "X123R" and canon("Ter123Arg") == "*123R"
assert canon("p.Arg1450Xaa") == "R1450Xaa"      # 미지정 아미노산은 종결로 읽지 않는다
_save = CFG["NORMALIZE_TOKENS"]; CFG["NORMALIZE_TOKENS"] = True
assert cell_tokens("R1450X R1450* p.Arg1450Ter") == ["R1450*"]          # 같은 변이의 여러 표기 -> 1개
assert cell_tokens("R175H, V600E") == ["R175H", "V600E"] and cell_tokens("WT") == []
CFG["NORMALIZE_TOKENS"] = _save


파싱 완료. 유형별 총 개수(train): {'missense': np.int64(161051), 'silent': np.int64(64844), 'nonsense': np.int64(13080), 'frameshift': np.int64(9767), 'splice': np.int64(0), 'inframe': np.int64(3), 'other': np.int64(319)}


### 표기 점검 (train 전용)
train 데이터의 변이 표기 현황을 확인한다. HGVS 표기 규약상 같은 변이가 여러 방식(`R1450X`, `R1450*`, `p.Arg1450Ter`)으로 적힐 수 있어, 입력 표기가 달라도 같은 변이로 취급하도록 정규화(`canon`)를 둔다.
정규화는 토큰 하나만 보고 바꾸는 행 독립 연산이며 train/test에 동일하게 적용된다.

**train 점검 결과**: 구식 종결 표기(`숫자+X`), `p.` 접두, 3글자 아미노산 표기는 train에 **0건**이다. 즉 이 정규화는 train 데이터로 필요성이 확인된 것이 아니라 HGVS 규약에 따른 방어적 처리이며, 적용 여부는 대회 운영진 확인 결과에 따른다 (`NORMALIZE_TOKENS` 플래그로 on/off, 제출 파일명에 norm/nonorm 표시).


In [8]:
# train에서 표기 혼재 여부를 확인한다 (test는 사용하지 않음)
def tokens_per_cell(df, normalize):
    """non-WT 셀 하나당 평균 변이 토큰 수 (normalize=True면 canon 적용 + 중복 제거)."""
    vals = df[gene_cols].to_numpy(dtype=object).ravel()
    vals = vals[vals != "WT"]
    n = [len(dict.fromkeys(canon(t) for t in SEP.split(str(v)) if t and t != "WT")) if normalize
         else len([t for t in SEP.split(str(v)) if t and t != "WT"]) for v in vals]
    return float(np.mean(n)) if n else 0.0

_vals = pd.Series(train[gene_cols].to_numpy(dtype=object).ravel())
_vals = _vals[_vals != "WT"]
_toks = pd.Series([t for v in _vals for t in SEP.split(str(v)) if t and t != "WT"])

x_end = _toks[_toks.str.contains(r"^(?:p\.)?[A-Z]\d+X$")]
print(f"train 전체 토큰 {len(_toks):,}개 중 '숫자+X' 종결 표기: {len(x_end)}개")
print(x_end.value_counts().head(10))
print("p. 접두 표기:", int(_toks.str.startswith("p.").sum()),
      "| 3글자 아미노산 표기:", int(_toks.str.contains(r"[A-Z][a-z]{2}\d|\d[A-Z][a-z]{2}").sum()))
print(f"train 칸당 변이 수 {tokens_per_cell(train, False):.2f} -> (통일·중복 제거 후) {tokens_per_cell(train, True):.2f}")


train 전체 토큰 255,164개 중 '숫자+X' 종결 표기: 0개
Series([], Name: count, dtype: int64)
p. 접두 표기: 0 | 3글자 아미노산 표기: 0
train 칸당 변이 수 1.17 -> (통일·중복 제거 후) 1.14


In [9]:
# ---- 경로/유전자군 사전 (초안: 필요에 맞게 수정/추가) ----
PATHWAYS = {
    "p53": ["TP53","MDM2","MDM4","CDKN2A","ATM","CHEK1","CHEK2"],
    "RTK_RAS": ["EGFR","ERBB2","ERBB3","ERBB4","FGFR1","FGFR2","FGFR3","FGFR4","MET","ALK","RET","KIT",
                "PDGFRA","KRAS","NRAS","HRAS","BRAF","RAF1","NF1","MAP2K1","MAP2K2","SOS1","PTPN11"],
    "PI3K": ["PIK3CA","PIK3CB","PIK3R1","PTEN","AKT1","AKT2","AKT3","MTOR","TSC1","TSC2","STK11","RICTOR","INPP4B"],
    "Wnt": ["APC","CTNNB1","AXIN1","AXIN2","RNF43","ZNRF3","TCF7L2","GSK3B","AMER1"],
    "CellCycle": ["RB1","CDK4","CDK6","CCND1","CCND2","CCND3","CCNE1","CDKN1A","CDKN1B","CDKN2B","E2F1","FBXW7"],
    "Notch": ["NOTCH1","NOTCH2","NOTCH3","NOTCH4","FBXW7","SPEN"],
    "Hippo": ["NF2","LATS1","LATS2","SAV1","YAP1","WWTR1","TEAD1"],
    "TGFb": ["TGFBR1","TGFBR2","SMAD2","SMAD3","SMAD4","ACVR1B","ACVR2A"],
    "NRF2": ["NFE2L2","KEAP1","CUL3"],
    "SWI_SNF": ["ARID1A","ARID1B","ARID2","SMARCA4","SMARCB1","PBRM1","SMARCA2"],
    "Chromatin": ["KMT2A","KMT2B","KMT2C","KMT2D","EZH2","SETD2","KDM6A","KDM5C","CREBBP","EP300",
                  "BAP1","ASXL1","DNMT3A","IDH1","IDH2","ATRX"],
    "DDR_MMR": ["BRCA1","BRCA2","PALB2","ATR","POLE","POLD1","MLH1","MSH2","MSH6","PMS2",
                "RAD51","RAD50","MRE11","FANCA","FANCD2","ERCC2"],
    "Splicing": ["SF3B1","SRSF2","U2AF1","RBM10"],
    "LongGenes": ["TTN","MUC16","MUC4","SYNE1","OBSCN","RYR2","RYR1","CSMD1","CSMD3","LRP1B","USH2A","FLG","PCLO"],
}
gene_pos = {g: i for i, g in enumerate(gene_cols)}
PW_IDX = {k: [gene_pos[g] for g in v if g in gene_pos] for k, v in PATHWAYS.items()}
print({k: f"{len(v)}/{len(PATHWAYS[k])}" for k, v in PW_IDX.items()})   # 데이터에 존재하는 유전자 수/사전 크기

{'p53': '5/7', 'RTK_RAS': '14/23', 'PI3K': '11/13', 'Wnt': '6/9', 'CellCycle': '12/12', 'Notch': '5/6', 'Hippo': '4/7', 'TGFb': '6/7', 'NRF2': '2/3', 'SWI_SNF': '0/7', 'Chromatin': '6/16', 'DDR_MMR': '11/16', 'Splicing': '2/4', 'LongGenes': '4/13'}


In [10]:
# ---- 유전자/토큰/아미노산쌍 사전은 "학습에 쓰는 행"에서만 만든다 (Data Leakage / CV 왜곡 방지) ----
# CV에서는 fold의 train 행만으로 사전을 만들고, 그 fold의 valid/test는 같은 사전으로 변환만 한다.
# test에만 있는 토큰은 tok_idx.get()이 None이라 무시된다.
def uniq_pairs(ex, in_rows):
    return set((r, nm) for r, nm in zip(ex["tok_rows"].tolist(), ex["tok_names"]) if in_rows[r])

def fit_vocab(ex, rows):
    rows = np.asarray(rows)
    in_rows = np.zeros(ex["mut"].shape[0], bool); in_rows[rows] = True
    keep_idx = np.where(ex["mut"][rows].sum(0) >= CFG["MIN_GENE_FREQ"])[0]
    cnt = Counter(nm for _, nm in uniq_pairs(ex, in_rows))
    keep_tokens = sorted(t for t, c in cnt.items() if c >= CFG["MIN_HOTSPOT_FREQ"])
    aa_keep = np.where(ex["aa_cnt"][rows].sum(0) >= CFG["MIN_AA_FREQ"])[0]
    return dict(keep_idx=keep_idx, keep_tokens=keep_tokens,
                tok_idx={t: i for i, t in enumerate(keep_tokens)}, aa_keep=aa_keep)

def compose(ex, voc):
    mut = ex["mut"]; n = mut.shape[0]
    blocks, names = [], []
    def add(a, nm):
        blocks.append(np.asarray(a, np.float32).reshape(n, -1)); names.extend(nm)

    add(mut[:, voc["keep_idx"]], [f"g_{gene_cols[i]}" for i in voc["keep_idx"]])

    n_genes = mut.sum(1)
    tc = ex["type_cnt"]; n_ev = tc.sum(1)
    add(np.c_[n_genes, np.log1p(n_genes), n_ev], ["n_mut_genes", "log_n_mut_genes", "n_events"])
    add(tc, [f"cnt_{t}" for t in TYPES])
    trunc = tc[:, [TYPE_IDX[t] for t in ("nonsense", "frameshift", "splice")]].sum(1)
    add(np.c_[tc[:, TYPE_IDX["silent"]] / (n_ev + 1),
              tc[:, TYPE_IDX["missense"]] / (n_ev + 1),
              trunc / (n_ev + 1)], ["ratio_silent", "ratio_missense", "ratio_trunc"])

    if CFG["USE_HOTSPOTS"] and voc["keep_tokens"]:
        H = np.zeros((n, len(voc["keep_tokens"])), np.float32)
        for r, nm in zip(ex["tok_rows"], ex["tok_names"]):
            j = voc["tok_idx"].get(nm)
            if j is not None:
                H[r, j] = 1
        add(H, [f"hs_{t}" for t in voc["keep_tokens"]])
    if CFG["USE_COMBOS"]:
        C, cn = combo_features(ex)
        add(C, [f"cb_{k}" for k in cn])
    if CFG["USE_AA_SUBS"] and len(voc["aa_keep"]):
        A = ex["aa_cnt"][:, voc["aa_keep"]] / (tc[:, TYPE_IDX["missense"]] + 1)[:, None]
        add(A, [f"aa_{AA[k // 20]}>{AA[k % 20]}" for k in voc["aa_keep"]])

    if CFG["USE_PATHWAYS"]:
        P = np.c_[[mut[:, idx].sum(1) if idx else np.zeros(n) for idx in PW_IDX.values()]].T
        add(P, [f"pw_{k}" for k in PW_IDX])

    return np.hstack(blocks), names

# 참고용(진단): 전체 train 기준 사전 크기. 모델 학습에는 쓰지 않고 fold마다 fit_vocab을 다시 호출한다.
_v = fit_vocab(ex_tr, np.arange(len(train)))
print(f"(참고) 전체 train 기준 유전자 {len(_v['keep_idx'])}/{len(gene_cols)}, 핫스팟 토큰 {len(_v['keep_tokens'])}, 아미노산쌍 {len(_v['aa_keep'])}")


(참고) 전체 train 기준 유전자 4171/4384, 핫스팟 토큰 1795, 아미노산쌍 158


### 4-0. 프로필 중복·라벨 충돌 진단 + 그룹 분할 준비

같은 환자가 개별 코호트(`KIRC`)와 통합 코호트(`KIPAN`)에 모두 등록되면 변이 프로필이 완전히 같은 행이 생긴다 (`LGG`↔`GBMLGG`도 동일). 이를 **후처리로 덮어쓰지 않고**, 먼저 규모를 진단하고 분할 단계에서 같은 프로필이 학습/검증에 갈라지지 않게 묶는다.
- 그룹 = 변이 프로필 해시. `StratifiedGroupKFold`로 클래스 비율은 유지하면서 같은 그룹은 같은 fold로 보낸다.
- 변이가 거의 없는 프로필(예: 전부 WT)은 우연히 같을 수 있으므로, 진단에서 변이 유전자 수 기준(`TMB_LOW`)으로 따로 센다.


In [11]:
import hashlib

def profile_hashes(df):
    """행마다 변이 프로필(유전자 셀 전부)의 지문. 셀이 전부 같으면 지문이 같다."""
    return np.array([hashlib.md5("|".join(r).encode()).hexdigest()[:12]
                     for r in df[gene_cols].to_numpy(dtype=object)])

H_tr, H_te = profile_hashes(train), profile_hashes(test)
lab_tr = train["SUBCLASS"].values

# ---- 진단: 같은 프로필 그룹과 라벨 충돌 ----
SIBLINGS = [frozenset(("KIRC", "KIPAN")), frozenset(("LGG", "GBMLGG"))]
g = pd.DataFrame({"h": H_tr, "y": lab_tr, "tmb": (train[gene_cols] != "WT").sum(1).values})
grp = g.groupby("h").agg(n=("y", "size"), labels=("y", lambda s: frozenset(s)), tmb=("tmb", "first"))
dup = grp[grp["n"] > 1]
conf = dup[dup["labels"].map(len) > 1]
sib = conf[conf["labels"].isin(SIBLINGS)]
print(f"train 프로필 그룹 {len(grp):,}개 중 2행 이상 중복: {len(dup):,}개 (해당 행 {int(dup['n'].sum()):,}개, 전체의 {dup['n'].sum()/len(train):.1%})")
print(f"  - 변이 유전자 {CFG['TMB_LOW']}개 이상인 중복 그룹(우연일 가능성 낮음): {int((dup['tmb'] >= CFG['TMB_LOW']).sum())}개")
print(f"  - 서로 다른 라벨이 섞인 그룹: {len(conf)}개 -> 형제 관계(KIRC/KIPAN, LGG/GBMLGG): {len(sib)}개, 그 외: {len(conf) - len(sib)}개")
if len(conf) - len(sib):
    print("  - 형제 외 라벨 충돌 예:", [tuple(sorted(s)) for s in conf[~conf['labels'].isin(SIBLINGS)]['labels'].head(5)])
in_tr = pd.Series(H_te).isin(set(H_tr))
print(f"test 행 중 train과 프로필이 같은 행: {int(in_tr.sum())}개 ({in_tr.mean():.1%})  <- 학습 데이터에 같은 프로필이 있다는 뜻 (정답 힌트로 쓰지 않음)")

# ---- 분할: 같은 프로필은 같은 fold로 ----
GRP_SIZE = pd.Series(H_tr).map(pd.Series(H_tr).value_counts()).to_numpy(dtype=float)   # 행마다 같은 프로필 행 수

def make_folds():
    if CFG["GROUP_CV"]:
        sp = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
        return list(sp.split(np.zeros(len(y)), y, groups=H_tr))
    sp = StratifiedKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
    return list(sp.split(np.zeros(len(y)), y))

FOLDS = make_folds()
if CFG["GROUP_CV"]:   # 같은 프로필이 학습/검증에 갈라지지 않았는지 확인
    assert all(not (set(H_tr[tr]) & set(H_tr[va])) for tr, va in FOLDS), "그룹 분할 실패: 같은 프로필이 train/valid에 걸쳐 있음"
print("분할:", "그룹(프로필) 분할" if CFG["GROUP_CV"] else "무작위 분할", "| fold 크기:", [len(va) for _, va in FOLDS])


train 프로필 그룹 5,636개 중 2행 이상 중복: 451개 (해당 행 1,016개, 전체의 16.4%)
  - 변이 유전자 5개 이상인 중복 그룹(우연일 가능성 낮음): 394개
  - 서로 다른 라벨이 섞인 그룹: 447개 -> 형제 관계(KIRC/KIPAN, LGG/GBMLGG): 445개, 그 외: 2개
  - 형제 외 라벨 충돌 예: [('LGG', 'THCA'), ('BRCA', 'CESC', 'GBMLGG', 'KIPAN', 'KIRC', 'LAML', 'LGG', 'LUSC', 'PAAD', 'PCPG', 'PRAD', 'SARC', 'STES', 'THCA', 'THYM')]
test 행 중 train과 프로필이 같은 행: 266개 (10.4%)  <- 학습 데이터에 같은 프로필이 있다는 뜻 (정답 힌트로 쓰지 않음)
분할: 그룹(프로필) 분할 | fold 크기: [1241, 1240, 1240, 1240, 1240]


## 4. 5-fold CV (XGBoost + Logistic Regression)

※ XGB는 검증 fold로 early stopping을 하므로 OOF 점수가 아주 조금 낙관적일 수 있습니다 (모델 간 비교용으로는 충분).
※ 유전자/핫스팟/아미노산쌍 선택(빈도 필터)은 fold의 train 행만으로 매 fold 다시 하고, valid/test에는 그 사전으로 변환만 한다.
※ 분할은 4-0절의 `FOLDS`(GROUP_CV면 같은 프로필끼리 묶음)를 쓴다. 이전 무작위 분할보다 점수가 낮아져도 그만큼 중복에 의한 낙관이 빠진 것이다.


In [12]:
def run_cv(ex_tr, ex_te, verbose=True):
    n_te = ex_te["mut"].shape[0]
    oof_xgb = np.zeros((len(y), n_class)); te_xgb = np.zeros((n_te, n_class))
    oof_lr = np.zeros((len(y), n_class));  te_lr = np.zeros((n_te, n_class))
    imp = Counter()   # 피처 이름별 중요도 (fold마다 피처 구성이 달라서 이름으로 합산)

    for fold, (tr, va) in enumerate(FOLDS):
        voc = fit_vocab(ex_tr, tr)                      # 사전은 이 fold의 train 행만으로
        X_all, names = compose(ex_tr, voc)              # 전체 train을 같은 사전으로 변환
        X_te, _ = compose(ex_te, voc)                   # test도 같은 사전으로 변환만
        X = X_all
        assert len(np.unique(y[tr])) == n_class, "일부 클래스가 train fold에 없습니다 (N_FOLDS를 줄이세요)"
        sw = compute_sample_weight("balanced", y[tr]) if CFG["BALANCE"] else None
        pw = None
        if CFG["PROFILE_WEIGHT"]:                       # 같은 프로필 행이 많을수록 행당 가중치를 낮춤
            pw = 1.0 / GRP_SIZE[tr]; pw = pw / pw.mean()
            sw = pw if sw is None else sw * pw

        m = xgb.XGBClassifier(
            objective="multi:softprob", n_estimators=CFG["XGB_N_EST"], learning_rate=0.08,
            max_depth=5, subsample=0.8, colsample_bytree=0.3, tree_method="hist",
            eval_metric="mlogloss", early_stopping_rounds=50, random_state=CFG["SEED"], n_jobs=-1,
        )
        m.fit(X[tr], y[tr], sample_weight=sw, eval_set=[(X[va], y[va])], verbose=False)
        oof_xgb[va] = m.predict_proba(X[va]); te_xgb += m.predict_proba(X_te) / CFG["N_FOLDS"]
        for nm_, v_ in zip(names, m.feature_importances_): imp[nm_] += v_ / CFG["N_FOLDS"]

        lr = make_pipeline(MaxAbsScaler(), LogisticRegression(
            C=CFG["LR_C"], max_iter=500, class_weight="balanced" if CFG["BALANCE"] else None))
        if pw is None:
            lr.fit(X[tr], y[tr])
        else:
            lr.fit(X[tr], y[tr], logisticregression__sample_weight=pw)
        oof_lr[va] = lr.predict_proba(X[va]); te_lr += lr.predict_proba(X_te) / CFG["N_FOLDS"]

        if verbose:
            f = lambda p: f1_score(y[va], p[va].argmax(1), average="macro")
            print(f"fold{fold} | XGB(best_iter={m.best_iteration}) macroF1={f(oof_xgb):.4f} | LR macroF1={f(oof_lr):.4f}")
    return oof_xgb, te_xgb, oof_lr, te_lr, imp

oof_xgb, te_xgb, oof_lr, te_lr, imp = run_cv(ex_tr, ex_te)


fold0 | XGB(best_iter=237) macroF1=0.4447 | LR macroF1=0.4219
fold1 | XGB(best_iter=252) macroF1=0.4783 | LR macroF1=0.4193
fold2 | XGB(best_iter=307) macroF1=0.4729 | LR macroF1=0.4266
fold3 | XGB(best_iter=300) macroF1=0.5070 | LR macroF1=0.4321
fold4 | XGB(best_iter=245) macroF1=0.4875 | LR macroF1=0.4438


In [13]:
# ---- (선택) 콤보 피처 효과 확인: CFG["RUN_ABLATION"]=True로 켜서 실행 ----
if CFG["RUN_ABLATION"]:
    keep_flag = CFG["USE_COMBOS"]
    CFG["USE_COMBOS"] = False
    o_x, _, o_l, _, _ = run_cv(ex_tr, ex_te, verbose=False)
    CFG["USE_COMBOS"] = keep_flag
    for nm, p in [("콤보 OFF  XGB", o_x), ("콤보 OFF  LR ", o_l), ("콤보 ON   XGB", oof_xgb), ("콤보 ON   LR ", oof_lr)]:
        print(f"{nm}  macroF1={f1_score(y, p.argmax(1), average='macro'):.4f}  acc={accuracy_score(y, p.argmax(1)):.4f}")
else:
    print("ablation 건너뜀 (CFG['RUN_ABLATION']=True로 실행 가능)")


ablation 건너뜀 (CFG['RUN_ABLATION']=True로 실행 가능)


## 5. 평가 · 앙상블 · 오분류 분석

In [14]:
def report(name, p):
    print(f"{name:12s} acc={accuracy_score(y, p.argmax(1)):.4f}  macroF1={f1_score(y, p.argmax(1), average='macro'):.4f}")

report("XGB", oof_xgb); report("LR", oof_lr)

best_w, best_s = 0.5, -1
for w in np.linspace(0, 1, 11):
    s = f1_score(y, (w * oof_xgb + (1 - w) * oof_lr).argmax(1), average="macro")
    if s > best_s: best_w, best_s = w, s
oof_ens = best_w * oof_xgb + (1 - best_w) * oof_lr
te_ens = best_w * te_xgb + (1 - best_w) * te_lr
print(f"\n앙상블 XGB 가중치={best_w:.1f}")
report("ENSEMBLE", oof_ens)

print("\n" + classification_report(y, oof_ens.argmax(1), target_names=le.classes_, digits=3))

cm = confusion_matrix(y, oof_ens.argmax(1)); np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], le.classes_[i], le.classes_[j]) for i in range(n_class) for j in range(n_class) if cm[i, j] > 0), reverse=True)[:10]
print("가장 많이 헷갈리는 쌍 (정답 -> 예측):")
for c, a, b in pairs: print(f"  {a} -> {b}: {c}")

print("\nXGB 중요도 상위 20:", [k for k, _ in imp.most_common(20)])

XGB          acc=0.4827  macroF1=0.4772
LR           acc=0.4304  macroF1=0.4324

앙상블 XGB 가중치=0.5
ENSEMBLE     acc=0.4919  macroF1=0.4943

              precision    recall  f1-score   support

         ACC      0.914     0.736     0.815        72
        BLCA      0.583     0.404     0.477       104
        BRCA      0.725     0.450     0.556       786
        CESC      0.356     0.432     0.391       155
        COAD      0.805     0.740     0.771       223
        DLBC      0.857     0.316     0.462        38
      GBMLGG      0.540     0.341     0.418       461
        HNSC      0.383     0.359     0.370       223
       KIPAN      0.406     0.346     0.374       515
        KIRC      0.418     0.428     0.423       334
        LAML      0.542     0.487     0.513       158
         LGG      0.470     0.721     0.569       229
        LIHC      0.473     0.392     0.429       158
        LUAD      0.466     0.337     0.391       184
        LUSC      0.547     0.584     0.565       1

In [15]:
# COAD 비율 보정: 검토 결과 미적용(k=1.0) → test 정답 분포 정보를 쓰지 않기 위해 삭제 (11/12)
te_final = te_ens.copy()


## 5-2. 쌍둥이(형제 라벨) 규칙 — 진단용 (`USE_TWIN`, 기본 OFF)

- TCGA의 **KIPAN**(= KIRC+KIRP+KICH)과 **GBMLGG**(= GBM+LGG)는 통합 코호트라, 같은 환자가 개별 코호트와 통합 코호트에 모두 들어 있을 수 있다 (변이 프로필 동일).
- 이전 버전은 train에 짝 라벨이 있는 프로필이 test에 나오면 짝 암종으로 **덮어썼다**. 이는 모델이 학습한 것이 아니라 분석가가 라벨 구조를 이용해 예측을 바꾸는 후처리다. 새 데이터에서 다른 형태의 중복이 나오면 무용지물이거나 오답을 내고, 대회의 "비정상적인 방법/Data Leakage" 조항에 걸릴 위험도 있다 (TwinRuleRefine.md).
- 그래서 근본 처리를 앞단으로 옮겼다: 4-0절의 **프로필 중복·충돌 진단**과 **그룹 분할 CV**. 이 셀의 규칙은 규모 확인용 진단으로만 남기고, 기본값은 적용하지 않는다.
- 그룹 분할(`GROUP_CV=True`)에서는 같은 프로필이 학습/검증에 갈라지지 않으므로 아래 시뮬의 발동 행이 0이다. 이것이 정상이며, 규칙의 이득을 OOF로는 검증할 수 없다는 뜻이다.
- 규칙을 굳이 쓰려면 `USE_TWIN=True`로 켜고, 대회 운영진에게 허용 여부를 먼저 확인한다 (제출 파일명에 `_twin`이 붙는다).


In [16]:
TWIN_SWAP = {"KIPAN": "KIRC", "KIRC": "KIPAN", "GBMLGG": "LGG", "LGG": "GBMLGG"}

def twin_override(tr_hash, tr_labels, q_hash):
    """질의 행마다 바꿀 암종 번호(없으면 -1). train 프로필->암종 집합 사전을 만들고 한 행씩 찾아본다."""
    lab = pd.Series(np.asarray(tr_labels)).groupby(np.asarray(tr_hash)).agg(set).to_dict()
    cls = list(le.classes_)
    out = np.full(len(q_hash), -1)
    for i, h in enumerate(q_hash):
        s = lab.get(h, set())
        if len(s) == 1:                      # 쌍이 train에 이미 둘 다 있으면(암종 2개) 건드리지 않음
            only = next(iter(s))
            if only in TWIN_SWAP and TWIN_SWAP[only] in cls:
                out[i] = cls.index(TWIN_SWAP[only])
    return out

def apply_override(P, ov):
    """ov >= 0 인 행은 그 암종 확률을 1로(나머지 0), 나머지 행은 그대로."""
    Q = P.copy(); m = ov >= 0
    Q[m] = 0.0; Q[m, ov[m]] = 1.0
    return Q

# ---- 동작 확인 (작은 예제) ----
le_bak = le
le = type("L", (), {"classes_": np.array(["BRCA", "GBMLGG", "KIPAN", "KIRC", "LGG"])})()
_ov = twin_override(["a","a","b","c","d","e","e"], ["KIPAN","KIRC","KIRC","LGG","BRCA","KIPAN","KIPAN"],
                    ["a","b","c","d","z","e"])
le = le_bak
assert _ov.tolist() == [-1, 2, 1, -1, -1, 3]   # a: 쌍이 이미 train에 있음 / b: KIRC->KIPAN / c: LGG->GBMLGG / e: KIPAN->KIRC
assert np.allclose(apply_override(np.full((6, 5), 0.2), _ov).sum(1), 1)

# ---- 시뮬(진단): CV와 같은 FOLDS, 학습 부분만 사전으로 사용 ----
sim = oof_ens.copy(); n_hit = 0
for tr_i, va_i in FOLDS:
    ov = twin_override(H_tr[tr_i], lab_tr[tr_i], H_tr[va_i])
    sim[va_i] = apply_override(oof_ens[va_i], ov); n_hit += int((ov >= 0).sum())
hit = (sim != oof_ens).any(1)
f0, f1 = (f1_score(y, p.argmax(1), average="macro") for p in (oof_ens, sim))
print(f"쌍둥이 규칙 시뮬: OOF macroF1 {f0:.4f} -> {f1:.4f} ({f1 - f0:+.4f}) | 발동 {n_hit}행 ({n_hit/len(y):.1%})")
if n_hit:
    print(f"  발동 행 정답률 {(sim[hit].argmax(1) == y[hit]).mean():.3f} (규칙 전 {(oof_ens[hit].argmax(1) == y[hit]).mean():.3f})")
elif CFG["GROUP_CV"]:
    print("  (그룹 분할이라 같은 프로필이 검증 fold에 짝으로 남지 않음 -> 시뮬 불가. 정상)")

# ---- test 쪽 규모 확인 (진단) ----
ov_te = twin_override(H_tr, lab_tr, H_te)
print(f"test에서 발동 가능했을 행: {(ov_te >= 0).sum()}행 ({(ov_te >= 0).mean():.1%}) ->",
      pd.Series(le.classes_[ov_te[ov_te >= 0]]).value_counts().to_dict())

twin_applied = False
if CFG["USE_TWIN"] and (f1 - f0) >= CFG["BOOST_MIN_GAIN"]:
    te_final = apply_override(te_final, ov_te); twin_applied = True
    print("쌍둥이 규칙 적용함 (운영진 허용 여부 확인 필요)")
else:
    print("쌍둥이 규칙 미적용 (기본값 USE_TWIN=False, 또는 시뮬 이득이 BOOST_MIN_GAIN 미만)")


쌍둥이 규칙 시뮬: OOF macroF1 0.4943 -> 0.4943 (+0.0000) | 발동 0행 (0.0%)
  (그룹 분할이라 같은 프로필이 검증 fold에 짝으로 남지 않음 -> 시뮬 불가. 정상)
test에서 발동 가능했을 행: 222행 (8.7%) -> {'KIRC': 64, 'KIPAN': 58, 'GBMLGG': 52, 'LGG': 48}
쌍둥이 규칙 미적용 (기본값 USE_TWIN=False, 또는 시뮬 이득이 BOOST_MIN_GAIN 미만)


## 6. 제출 파일 생성

In [17]:
pred = le.inverse_transform(te_final.argmax(1))
out = sub.copy()
out["SUBCLASS"] = pred
assert len(out) == len(test) and set(out["SUBCLASS"]) <= set(le.classes_)

# tag 플래그를 이용해 norm/nonorm 버전 파일명 자동 분기
tag = ("norm" if CFG["NORMALIZE_TOKENS"] else "nonorm") + ("_twin" if twin_applied else "")
out.to_csv(f"{D}/submission_v8_{tag}.csv", index=False, encoding="UTF-8-sig")
np.save(f"{D}/oof_ens_v8_{tag}.npy", oof_ens)
np.save(f"{D}/te_final_v8_{tag}.npy", te_final)

print(out["SUBCLASS"].value_counts().head(10))

SUBCLASS
COAD      519
STES      419
KIPAN     198
BRCA      141
OV        118
PRAD      102
KIRC       91
GBMLGG     89
SKCM       85
LGG        78
Name: count, dtype: int64
